# EmotionLens — Quick-Start Notebook

This notebook walks through the core features of EmotionLens:
- Single-sentence text prediction
- Batch analysis
- VAD space visualisation
- Fusion strategy comparison


In [ ]:
# Install emotionlens if not already installed
# !pip install -e ..

from emotionlens import EmotionPipeline
pipe = EmotionPipeline(explain=True)
print('Pipeline ready!')

## 1. Single prediction

In [ ]:
result = pipe.predict(text="I just got the promotion I've been working for!")
print(result)

## 2. Batch analysis

In [ ]:
sentences = [
    "I can't believe how amazing this is!",
    "Everything is falling apart.",
    "How dare they do this to me!",
    "I heard something in the dark...",
    "Oh wow, I had no idea!",
    "That smell is absolutely revolting.",
    "Whatever, I don't care.",
    "Just another regular Tuesday.",
]

results = pipe.batch_predict(sentences, show_progress=False)
for sentence, r in zip(sentences, results):
    print(f'{r.label.value:<12} ({r.confidence:.0%})  "{sentence[:55]}"')

## 3. VAD space visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

colors = {
    'joy': '#FFD700', 'sadness': '#4169E1', 'anger': '#DC143C',
    'fear': '#9370DB', 'surprise': '#00CED1', 'disgust': '#228B22',
    'contempt': '#808080', 'neutral': '#A0A0A0',
}

fig, ax = plt.subplots(figsize=(8, 6))
for sentence, r in zip(sentences, results):
    v, a = r.vad.valence, r.vad.arousal
    c = colors.get(r.label.value, 'gray')
    ax.scatter(v, a, c=c, s=200, zorder=3, label=r.label.value)
    ax.annotate(sentence[:25]+'…', (v, a), xytext=(5, 5),
                textcoords='offset points', fontsize=7)

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Valence (negative ← → positive)')
ax.set_ylabel('Arousal (calm ← → excited)')
ax.set_title('Emotion predictions in VAD space')
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)

# Quadrant labels
for x, y, text in [(-0.8, 0.8, 'Angry\nFearful'), (0.8, 0.8, 'Happy\nExcited'),
                    (-0.8, -0.8, 'Sad\nDepressed'), (0.8, -0.8, 'Content\nRelaxed')]:
    ax.text(x, y, text, ha='center', va='center', fontsize=9,
            color='gray', style='italic')

plt.tight_layout()
plt.savefig('vad_space.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Fusion strategy comparison

In [ ]:
test_text = "I'm so worried but also strangely excited."
print(f'Text: "{test_text}"\n')

for strategy in ['weighted_average', 'confidence_gating', 'attention']:
    p = EmotionPipeline(fusion_strategy=strategy, explain=False)
    r = p.predict(text=test_text)
    print(f'Strategy: {strategy:<22}  → {r.label.value:<10} {r.confidence:.0%}')

## 5. Score distribution heatmap

In [ ]:
import pandas as pd

score_rows = []
for sentence, r in zip(sentences, results):
    row = {'text': sentence[:35]}
    row.update(r.all_scores)
    score_rows.append(row)

df = pd.DataFrame(score_rows).set_index('text')

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(df.columns)))
ax.set_xticklabels(df.columns, rotation=30, ha='right')
ax.set_yticks(range(len(df.index)))
ax.set_yticklabels(df.index, fontsize=8)
plt.colorbar(im, ax=ax, label='Score')
ax.set_title('Emotion score heatmap')
plt.tight_layout()
plt.savefig('score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()